## 3.1 긴 시퀀스 모델링의 문제점

![3.1_1](img/3.1_1.png)
- 한 단어씩 번역 시 번역 시 문법 오류 발생 가능
    - 번역 수행 위해 문맥 이해 및 올바른 문법적 결과 필요
    - => 인코더 & 디코더 사용

- 인코더-디코더 RNN 제약사항
    - 디코딩 단계에서 RNN이 인코더의 이전 은닉 사태 참조 불가능
    - -> 결과적으로 모든 관련된 정보가 포함된 최종 은닉 상태에만 의존 -> 맥락 놓치기 쉬움

## 3.2 어텐션 매커니즘으로 데이터 의존성 포착하기

- 어텐션 매커니즘
    - ![3-5](img/3-5.png)
    - 인코더-디코더 RNN 수정 -> 디코딩 단계마다 디코더가 선택적으로 입력 시퀀스의 서로 다른 부분 참조 가능

- **셀프 어텐션 매커니즘**
    - 시퀀스 표현 계산 시 입력 시퀀스에 있는 각 위치가 동일 시퀀스에 있는 다른 모든 위치와의 관련성을 고려 / 주의 기울일 수 있음

## 3.3 셀프 어텐션으로 입력의 서로 다른 부분에 주의 기울이기

### 3.3.1 훈련 가능한 가중치가 없는 간단한 셀프 어텐션 매커니즘

- 셀프 어텐션 목표: 다른 모든 입력 원소의 정보 조합 -> 각각의 입력 원소에 대한 문맥 벡터 계산
    - 목표: 입력 시퀀스에 있는 각 원소 $x^{(i)}$에 대한 문맥 벡터 $z^{(i)}$ 계산
        - ![3-7](img/3-7.png)
- 문맥 벡터 계산 위한 각 입력 원소의 중요도/기여도는 어텐션 가중치에 의해 결정

In [87]:
# 3차원 벡터로 임베딩된 입력 시퀀스 가정

import torch
inputs = torch.tensor(
    [[0.43, 0.15, 0.89],    # Your
     [0.55, 0.87, 0.66],    # journey
     [0.57, 0.85, 0.64],    # starts
     [0.22, 0.58, 0.33],    # with
     [0.77, 0.25, 0.10],    # one
     [0.05, 0.80, 0.55]]    # step
)

In [88]:
# 어텐션 점수 계산

query = inputs[1]   # 두번째 입력 토큰을 쿼리 토큰으로 사용
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)
print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


- 정규화 하는 이유
    - 해석 용이
    - LLM 훈련 시 안정성 유지에 도움

In [89]:
# 어텐션 점수 정규화

attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("어텐션 가중치:", attn_weights_2_tmp)
print("합:", attn_weights_2_tmp.sum())   # 가중치 합 1

어텐션 가중치: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
합: tensor(1.0000)


In [90]:
# 어텐션 점수 정규화하는 소프트맥스 함수 기본 구현
# 어텐션 가중치 항상 1 -> 출력 학률/상대적 중요도로 해석 가능

def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)
print("어텐션 가중치:", attn_weights_2_naive)
print("합:", attn_weights_2_naive.sum())

어텐션 가중치: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
합: tensor(1.)


In [91]:
# 파이토치 소프트맥스 사용
# 위 소프트맥스 구현의 수치 불안정 문제 해소

attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("어텐션 가중치:", attn_weights_2)
print("합:", attn_weights_2.sum())

어텐션 가중치: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
합: tensor(1.)


In [92]:
# 문맥 벡터: 모든 입력 벡터의 가중치 합
# 각 입력 벡터: 어텐션 가중치의 곱

query = inputs[1]   # 두번째 입력 토큰이 쿠러ㅣ
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * x_i
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


### 3.3.2 모든 입력 토큰에 대해 어텐션 가중치 계산하기

In [93]:
# 특정(2번째) 원소 뿐 아니라 모든 문맥 벡터 계산 위한 코드
# 정규화되지 않음

attn_scores = torch.empty(6, 6)
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [94]:
# for 느림 -> 행렬곱셈 사용

attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [95]:
# 정규화

attn_weights = torch.softmax(attn_scores, dim=1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [96]:
# 각 행의 값을 더해 1이 되는지 확인
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print("두 번째 행의 합:", row_2_sum)
print("모든 행의 합:", attn_weights.sum(dim=1))

두 번째 행의 합: 1.0
모든 행의 합: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [97]:
# 어텐션 가중치 & 입력 행렬곱 -> 문맥 벡터 계산
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


In [98]:
print("앞서 계산한 두 번째 문맥 벡터:", context_vec_2)

앞서 계산한 두 번째 문맥 벡터: tensor([0.4419, 0.6515, 0.5683])


## 3.4 훈련 가능한 가중치를 가진 셀프 어텐션 구현하기

### 3.4.1 단계별로 어텐션 가중치 계산하기

- 쿼리(q), 키(k), 벡터(v) 계산
    - ![3-14](img/3-14.png)

In [99]:
# 하나의(2번째) 문맥벡터 계산

x_2 = inputs[1]         # 두 번째 입력 원소
d_in = inputs.shape[1]  # 입력 임베딩 크기, d_in = 3
d_out = 2               # 출력 임베딩 크기, d_out = 2

In [100]:
# 가중치행렬 초기화

# 모델 훈련 시 requres_grad=True 지정 -> 훈련 과정에서 가중치 행렬 업데이트 필요
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

- 쿼리 & 키 사용해 어텐션 점수 계산
    - ![3-15](img/3-15.png)

In [101]:
# 쿼리, 키, 값 벡터 계산

query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value
print(query_2)

tensor([0.4306, 1.4551])


In [102]:
# 모든 입력 원소에 대한 키, 값 구하기
keys = inputs @ W_key
values = inputs @ W_value
print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


In [103]:
# 어텐션 점수 w_22 계산
keys_2 = keys[1]
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

tensor(1.8524)


In [104]:
# 행렬곱 이용해 모든 어텐션 점수 계산
attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


In [105]:
# 어텐션 가중치 계산
# 어텐션 점수 소프트맥스 함수로 정규화 -> 어텐션 가중치

d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k ** 0.5, dim=-1)
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


- 문맥 벡터 계산
    - 어텐션 가중치가 각각의 값 벡터의 중요도에 대한 가중치로 동작
    - ![3-17](img/3-17.png)

In [106]:
# 문맥 벡터 계산

context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])


### 3.4.2 셀프 어텐션 파이썬 클래스 구현하기

In [107]:
# 셀프 어텐션 클래스

import torch.nn as nn
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))    # Query 가중치
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))      # Key 가중치
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))    # value 가중치

    def forward(self, x):       # 순전파 함수
        keys = x @ self.W_key           # 입력으로부터 key 계산
        queries = x @ self.W_query      # 입력으로부터 query 계산
        values = x @ self.W_value       # 입력으로부터 value 계산
        attn_scores = queries @ keys.T  # query와 key의 유사도 계산
        attn_weights = torch.softmax(   # 유사도 정규화
            attn_scores / keys.shape[-1] ** 0.5, dim=-1
        )
        context_vec = attn_weights @ values
        return context_vec

In [108]:
# 위 클래스 사용

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [109]:
# 파이토치 Linear 층을 사용한 셀프 어텐션 클래스

class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qky_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qky_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qky_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qky_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        context_vec = attn_weights @ values
        return context_vec

In [110]:
# SelfAttention_v2 사용
# 가중치 행렬의 초깃값이 다르기 때문에 출력 결과 다름

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


In [129]:
"""연습문제 3.1"""

torch.manual_seed(123)

sa_v1 = SelfAttention_v1(d_in, d_out)
sa_v2 = SelfAttention_v2(d_in, d_out)

# v2 -> v1 로 복사
with torch.no_grad():
    sa_v1.W_query.copy_(sa_v2.W_query.weight.T)
    sa_v1.W_key.copy_(sa_v2.W_key.weight.T)
    sa_v1.W_value.copy_(sa_v2.W_value.weight.T)

out_v1 = sa_v1(inputs)
out_v2 = sa_v2(inputs)

print("SelfAttention_v1 출력:\n", out_v1)
print("SelfAttention_v2 출력:\n", out_v2)
print("두 출력이 같은가?:", torch.allclose(out_v1, out_v2))

SelfAttention_v1 출력:
 tensor([[0.5085, 0.3508],
        [0.5084, 0.3508],
        [0.5084, 0.3506],
        [0.5074, 0.3471],
        [0.5076, 0.3446],
        [0.5077, 0.3493]], grad_fn=<MmBackward0>)
SelfAttention_v2 출력:
 tensor([[0.5085, 0.3508],
        [0.5084, 0.3508],
        [0.5084, 0.3506],
        [0.5074, 0.3471],
        [0.5076, 0.3446],
        [0.5077, 0.3493]], grad_fn=<MmBackward0>)
두 출력이 같은가?: True


## 3.5 코잘 어텐션으로 미래의 단어를 감추기

- **코잘 어텐션(마스크드 어텐션, masked attention)**: 메커니즘이 주어진 토큰으로 어텐션 점수 계산 시퀀스의 이전 입력 & 현재 입력만 참조하도록 만듦
    - ![3-19](img/3-19.png)

### 3.5.1 코잘 어텐션 마스크 적용하기

- (정규화되지 않은) 어텐션 점수
    1. 소프트맥스 함수 적용
- (정규화된) 어텐션 가중치
    - 정규화: 각 행의 값 더하면 1이 됨
    2. 주대각선 위의 값을 0으로 만듦
- (정규화되지 않은) 마스킹된 어텐션 점수
    3. 각 행 정규화
- (정규화된) 마스킹된 어텐션 가중치

In [112]:
# 소프트맥스 함수 사용해 어텐션 가중치 계산

queries = sa_v2.W_query(inputs)     # 편의상 이전 SelfAttention_v2 객체의 쿼리 키 가중치행렬 재사용
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[0.1362, 0.1730, 0.1736, 0.1713, 0.1792, 0.1666],
        [0.1359, 0.1730, 0.1735, 0.1716, 0.1790, 0.1670],
        [0.1366, 0.1729, 0.1734, 0.1714, 0.1788, 0.1669],
        [0.1493, 0.1701, 0.1704, 0.1697, 0.1732, 0.1674],
        [0.1589, 0.1690, 0.1692, 0.1667, 0.1712, 0.1649],
        [0.1408, 0.1715, 0.1718, 0.1717, 0.1758, 0.1684]],
       grad_fn=<SoftmaxBackward0>)


In [113]:
# 파이토치의 tril 함수로 주대각선 위의 값이 0인 마스크 생성

context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [114]:
# 마스크와 어텐션 가중치 곱해 주대각선 위의 값 0으로 만듦

mask_simple = attn_weights * mask_simple
print(mask_simple)

tensor([[0.1362, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1359, 0.1730, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1366, 0.1729, 0.1734, 0.0000, 0.0000, 0.0000],
        [0.1493, 0.1701, 0.1704, 0.1697, 0.0000, 0.0000],
        [0.1589, 0.1690, 0.1692, 0.1667, 0.1712, 0.0000],
        [0.1408, 0.1715, 0.1718, 0.1717, 0.1758, 0.1684]],
       grad_fn=<MulBackward0>)


In [115]:
# 어텐션 가중치를 합이 1이 되도록 다시 정규화

row_sums = mask_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = mask_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4400, 0.5600, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2830, 0.3580, 0.3590, 0.0000, 0.0000, 0.0000],
        [0.2264, 0.2579, 0.2583, 0.2574, 0.0000, 0.0000],
        [0.1903, 0.2024, 0.2026, 0.1997, 0.2051, 0.0000],
        [0.1408, 0.1715, 0.1718, 0.1717, 0.1758, 0.1684]],
       grad_fn=<DivBackward0>)


In [116]:
# 더 효율적으로 마스킹

mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[-0.2327,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.2396,  0.1015,    -inf,    -inf,    -inf,    -inf],
        [-0.2323,  0.1004,  0.1045,    -inf,    -inf,    -inf],
        [-0.1344,  0.0502,  0.0523,  0.0470,    -inf,    -inf],
        [-0.0349,  0.0520,  0.0538,  0.0331,  0.0708,    -inf],
        [-0.2142,  0.0650,  0.0679,  0.0668,  0.1004,  0.0395]],
       grad_fn=<MaskedFillBackward0>)


In [117]:
# 마스킹된 결과에 소프트맥스 함수 적용

attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4400, 0.5600, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2830, 0.3580, 0.3590, 0.0000, 0.0000, 0.0000],
        [0.2264, 0.2579, 0.2583, 0.2574, 0.0000, 0.0000],
        [0.1903, 0.2024, 0.2026, 0.1997, 0.2051, 0.0000],
        [0.1408, 0.1715, 0.1718, 0.1717, 0.1758, 0.1684]],
       grad_fn=<SoftmaxBackward0>)


### 3.5.2 드롭아웃으로 어텐션 가중치에 추가적으로 마스킹하기

- **드롭아웃**(dropout): 훈련 중 유닛을 랜덤하게 선택 -> 해당 유닛의 출력 무시하는 기법
    - 모델이 은닉층의 특정 유닛에 과도하게 의존하지 않도록 함 => 과대적합 방지
    - 훈련 중에만 사용, 그 이후 비활성화
    - ![3-22](img/3-22.png)

In [118]:
# 드롭아웃 비율 50% 지정
# 6x6 텐서에 파이토치 드롭아웃 층 적용

torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5)     # 드롭아웃 비율 50% 지정
example = torch.ones(6, 6)          # 1로 채워진 행렬 생성
print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [119]:
# 어첸션 가중치 행렬에 드롭아웃 적용

torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5659, 0.7160, 0.7181, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.5159, 0.5167, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4047, 0.0000, 0.3993, 0.0000, 0.0000],
        [0.0000, 0.3430, 0.3437, 0.3434, 0.3516, 0.0000]],
       grad_fn=<MulBackward0>)


### 3.5.3 코잘 어텐션 클래스 구현하기

In [120]:
# 배치 입력 시뮬레이션

batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)      # 각각 6개의 토큰으로 구성된 2개의 입력, 각 토큰 임베딩 차원 3

torch.Size([2, 6, 3])


In [121]:
# 코잘 어텐션 클래스

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)  # SelfAttention_v1 클래스와 달리 드롭아웃 추가
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)

        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec

In [122]:
# CausalAttention 클래스 사용

torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
print("context_vecs.shape", context_vecs.shape)

context_vecs.shape torch.Size([2, 6, 2])


## 3.6 싱글 헤드 어텐션을 멀티 헤드 어텐션으로 확장하기

- **멀티 헤드**: 어텐션 메커니즘을 독립적으로 동작하는 여러 개의 헤드로 나누었다는 의미

### 3.6.1 여러 개의 싱글 헤드 어텐션 층 쌓기

- 멀티 헤드 어텐션 모듈 구조
    - ![3-24](img/3-24.png)

In [123]:
# 멀티 헤드 어텐션 클래스

class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length,
                 dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(
                d_in, d_out, context_length, dropout, qkv_bias
            )
            for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

In [124]:
# 사용

torch.manual_seed(123)

context_length = batch.shape[1]      # 토큰 개수
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(
    d_in, d_out, context_length, 0.0, num_heads=2
)
context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

"""
context_vecs.shape: torch.Size([2, 6, 4]
2개의 입력 텍스트 사용 -> 첫번째 차원 2
각 입력에 6개의 토큰 -> 두번째 차원 6
각 토큰이 4차원 임베딩 벡터(num_heads=2, d_out=2) -> 세번째 차원 4
"""

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


'\ncontext_vecs.shape: torch.Size([2, 6, 4]\n2개의 입력 텍스트 사용 -> 첫번째 차원 2\n각 입력에 6개의 토큰 -> 두번째 차원 6\n각 토큰이 4차원 임베딩 벡터(num_heads=2, d_out=2) -> 세번째 차원 4\n'

In [130]:
"""연습문제 3.2"""

d_in, d_out = 3, 1
mha = MultiHeadAttentionWrapper(
    d_in, d_out, context_length, 0.0, num_heads=2
)
context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.0189, 0.2729],
         [0.2181, 0.3037],
         [0.2804, 0.3125],
         [0.2830, 0.2793],
         [0.2476, 0.2541],
         [0.2748, 0.2513]],

        [[0.0189, 0.2729],
         [0.2181, 0.3037],
         [0.2804, 0.3125],
         [0.2830, 0.2793],
         [0.2476, 0.2541],
         [0.2748, 0.2513]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


### 3.6.2 가중치 분할로 멀티 헤드 어텐션 구현하기

In [125]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out,
                 context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out은 num_heads로 나누어 떨어져야 합니다"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads      # 원하는 출력 차원에 맞도록 투영 차원 나눔

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear 층을 사용해 헤드의 출력을 결합
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # 텐서 크기: (b, num_tokens, d_out)
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # num_heads 차원 ㅜ가 -> 암묵적으로 행렬 분할, 마지막 차원을 num_heads에 맞춰 채움
        # (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(
            b, num_tokens, self.num_heads, self.head_dim
        )

        # (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)    # 각 헤드애 대해 점곱 수행
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]  # 토큰 개수로 마스크 자름

        attn_scores.masked_fill_(mask_bool, -torch.inf)     # 마스크 사용해 어텐션 점수 채움

        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # 텐서 크기: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        context_vec = context_vec.contiguous().view(
            b, num_tokens, self.d_out
        )
        context_vec = self.out_proj(context_vec)    # 선형 투영 추가
        return context_vec

- ![3-26](img/3-26.png)
    - 위: MultiHeadAttentionWrapper 클래스에서는 2개의 가중치 행렬 $W_{q1}$과 $W_{q2}$를 초기화한 후 2개의 쿼리 행렬 $Q_1$ & $Q_2$ 계산
    - 아래: MultiHeadAttention 클래스에서는 큰 가중치 행렬 $W_q$ 하나를 초기화 -> 입력 & 행렬 곱셈 한번 더 수행해 쿼리행렬 $Q$ 얻음 -> 이 쿼리 행렬을 $Q_1$ & $Q_2$로 나눔

In [126]:
a = torch.tensor([[[[0.2745, 0.6584, 0.2775, 0.8573],
                    [0.8993, 0.0390, 0.9268, 0.7388],
                    [0.7179, 0.7058, 0.9156, 0.4340]],

                   [[0.0772, 0.3565, 0.1479, 0.5331],
                    [0.4066, 0.2318, 0.4545, 0.9737],
                    [0.4606, 0.5159, 0.4220, 0.5786]]]])

print(a @ a.transpose(2, 3))

tensor([[[[1.3208, 1.1631, 1.2879],
          [1.1631, 2.2150, 1.8424],
          [1.2879, 1.8424, 2.0402]],

         [[0.4391, 0.7003, 0.5903],
          [0.7003, 1.3737, 1.0620],
          [0.5903, 1.0620, 0.9912]]]])


In [127]:
# 행렬 곱셈 간편

first_head = a[0, 0, :, :]
first_res = first_head @ first_head.T
print("첫 번째 헤드:\n", first_res)

second_head = a[0, 1, :, :]
second_res = second_head @ second_head.T
print("\n두 번째 헤드: \n", second_res)

첫 번째 헤드:
 tensor([[1.3208, 1.1631, 1.2879],
        [1.1631, 2.2150, 1.8424],
        [1.2879, 1.8424, 2.0402]])

두 번째 헤드: 
 tensor([[0.4391, 0.7003, 0.5903],
        [0.7003, 1.3737, 1.0620],
        [0.5903, 1.0620, 0.9912]])


In [128]:
# 클래스 사용

torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


In [133]:
"""연습문제 3.3"""

torch.manual_seed(123)

batch = torch.randn(2, 1024, 768)

context_length = 1024
d_in = 768
d_out = 768
num_heads = 12

mha = MultiHeadAttention(
    d_in=d_in,
    d_out=d_out,
    context_length=context_length,
    dropout=dropout,
    num_heads=num_heads
)

context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-1.6569e-01, -2.5124e-01,  5.1141e-01,  ..., -7.5648e-02,
          -4.9951e-02, -4.2609e-01],
         [-3.0240e-01, -1.5263e-01,  9.3262e-02,  ..., -1.9805e-01,
           2.3410e-01, -5.2786e-01],
         [-1.6716e-01, -2.9942e-01,  3.0368e-02,  ..., -2.0547e-02,
           1.6848e-01, -1.9917e-01],
         ...,
         [ 2.4924e-02,  5.0410e-03,  2.4708e-02,  ...,  2.7143e-02,
          -8.8932e-03, -3.6202e-02],
         [ 2.4997e-02,  1.2060e-02,  8.1538e-03,  ...,  2.8854e-02,
          -2.4592e-02, -2.7696e-02],
         [ 3.5210e-02, -1.9875e-03,  1.4529e-02,  ...,  2.5048e-02,
          -2.2903e-02, -2.5456e-02]],

        [[ 7.1096e-02, -1.8721e-01, -3.5003e-01,  ...,  1.9014e-01,
           1.8846e-01, -5.5657e-01],
         [ 4.3102e-01, -1.5199e-01, -4.5988e-01,  ..., -2.0671e-02,
          -4.1174e-02, -3.7485e-01],
         [ 3.8715e-01,  3.9501e-02, -1.9719e-01,  ..., -4.9379e-02,
          -2.3517e-03, -7.3573e-02],
         ...,
         [ 1.9406e-02, -3